In [5]:
import requests
import pandas as pd
import time



In [ ]:
TMDB_API_KEY = ""
BASE_URL = "https://api.themoviedb.org/3"

In [ ]:
# send a get request to the tmdb api
def tmdb_get(endpoint, params=None, timeout=10):
    
    if params is None:
        params = {}
    params["api_key"] = TMDB_API_KEY

    url = BASE_URL + endpoint
    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()

In [ ]:
# collect movies for one release year
def collect_movies(year, pages=3, sort_by="popularity.desc", language="en-US", sleep=0.2):
    all_movies = []

    for page in range(1, pages + 1):

        data = tmdb_get(
            "/discover/movie",
            params={
                "primary_release_year": year,
                "sort_by": sort_by,
                "page": page,
                "language": language,
            }
        )
        all_movies.extend(data["results"])
        time.sleep(sleep)
        
    return pd.DataFrame(all_movies)

In [ ]:
# get additional movie details
def get_movie_details(movie_id):

    data = tmdb_get(f"/movie/{movie_id}")

    return {
        "id": data.get("id"),
        "budget": data.get("budget"),
        "revenue": data.get("revenue"),
        "runtime": data.get("runtime"),
        "genres": [g["name"] for g in data.get("genres", [])]
    }

In [ ]:
years = range(1990, 2025)
frames = []

for year in years:
    frames.append(collect_movies(year, pages=20))

df_movies = pd.concat(frames, ignore_index=True)
df_movies = df_movies.drop_duplicates("id")

In [ ]:
details = []

for movie_id in df_movies["id"]:

    try:
        d = get_movie_details(movie_id)
        details.append(d)

    except requests.exceptions.RequestException:
        continue

    time.sleep(0.2)
df_details = pd.DataFrame(details)

In [19]:
df_full = df_movies.merge(df_details, on ="id", how="left")


In [23]:
df_full.to_csv("movies_dataset.csv", index=False)